In [1]:
import heapq

def a_star(graph, heuristic, start, goal):
    """
    graph: {
        'A': {'B': 2, 'C': 4},
        'B': {'A': 2, 'D': 3, 'E': 5},
        ...
    }

    heuristic: {
        'A': 7,
        'B': 6,
        ...
    }

    start: đỉnh bắt đầu
    goal: đỉnh đích
    """

    open_list = []
    heapq.heappush(open_list, (0, start))

    g = {node: float('inf') for node in graph}
    g[start] = 0

    parent = {start: None}

    while open_list:
        f_current, current = heapq.heappop(open_list)

        if current == goal:
            path = []
            while current is not None:
                path.append(current)
                current = parent[current]
            path.reverse()
            return path, g[goal]

        for neighbor, cost in graph[current].items():
            tentative_g = g[current] + cost

            if tentative_g < g[neighbor]:
                g[neighbor] = tentative_g
                parent[neighbor] = current
                f_neighbor = tentative_g + heuristic[neighbor]
                heapq.heappush(open_list, (f_neighbor, neighbor))

    return None, float('inf')


# =========================
# Chương trình chính
# =========================
if __name__ == "__main__":
    graph = {
        'A': {'B': 6, 'F': 3},
        'B': {'A': 6, 'C': 3, 'D': 2},
        'C': {'B': 3, 'D': 1, 'E': 5},
        'D': {'B': 2, 'C': 1, 'E': 8},
        'E': {'C': 5, 'D': 8, 'I': 5, 'J': 5},
        'F': {'A': 3, 'G': 1, 'H': 7},
        'G': {'F': 1, 'I': 3},
        'H': {'F': 7, 'I': 2},
        'I': {'G': 3, 'H': 2, 'E': 5, 'J': 3},
        'J': {'E': 5, 'I': 3}
    }

    # Heuristic: giá trị ước lượng từ mỗi đỉnh đến đích J
    heuristic = {
        'A': 10,
        'B': 8,
        'C': 5,
        'D': 7,
        'E': 3,
        'F': 6,
        'G': 5,
        'H': 4,
        'I': 2,
        'J': 0
    }

    start = input("Nhập đỉnh bắt đầu: ").strip()
    goal = input("Nhập đỉnh đích: ").strip()

    if start not in graph or goal not in graph:
        print("Đỉnh nhập không tồn tại trong đồ thị.")
    else:
        path, cost = a_star(graph, heuristic, start, goal)

        if path is None:
            print("Không tìm thấy đường đi.")
        else:
            print("Đường đi ngắn nhất:", " -> ".join(path))
            print("Tổng chi phí:", cost)

Nhập đỉnh bắt đầu: A
Nhập đỉnh đích: J
Đường đi ngắn nhất: A -> F -> G -> I -> J
Tổng chi phí: 10


In [2]:
import heapq

class State:
    def __init__(self, current_city, visited, path, g, h):
        self.current_city = current_city
        self.visited = visited
        self.path = path
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, other):
        return self.f < other.f


def heuristic(current_city, visited, graph, all_cities, start_city):
    """
    Hàm ước lượng đơn giản:
    - lấy chi phí nhỏ nhất từ current_city đến một thành phố chưa thăm
    - cộng thêm chi phí nhỏ nhất để quay về start
    """
    unvisited = [city for city in all_cities if city not in visited]

    if not unvisited:
        return graph[current_city][start_city]  # quay về kho

    min_to_unvisited = min(graph[current_city][city] for city in unvisited)

    # tổng thêm một ước lượng nhẹ: mỗi thành phố chưa thăm sẽ cần ít nhất 1 cạnh nhỏ nhất
    extra = 0
    for city in unvisited:
        min_edge = min(graph[city][neighbor] for neighbor in graph[city] if neighbor != city)
        extra += min_edge

    return min_to_unvisited + extra // 2


def a_star_delivery(graph, start_city):
    all_cities = list(graph.keys())
    initial_visited = frozenset([start_city])

    open_list = []
    start_state = State(
        current_city=start_city,
        visited=initial_visited,
        path=[start_city],
        g=0,
        h=heuristic(start_city, initial_visited, graph, all_cities, start_city)
    )
    heapq.heappush(open_list, start_state)

    best_cost = {}

    while open_list:
        current = heapq.heappop(open_list)

        state_key = (current.current_city, current.visited)
        if state_key in best_cost and best_cost[state_key] <= current.g:
            continue
        best_cost[state_key] = current.g

        # Nếu đã đi qua tất cả thành phố thì quay về điểm xuất phát
        if len(current.visited) == len(all_cities):
            total_cost = current.g + graph[current.current_city][start_city]
            final_path = current.path + [start_city]
            return final_path, total_cost

        for neighbor in all_cities:
            if neighbor not in current.visited:
                new_visited = frozenset(set(current.visited) | {neighbor})
                new_g = current.g + graph[current.current_city][neighbor]
                new_path = current.path + [neighbor]
                new_h = heuristic(neighbor, new_visited, graph, all_cities, start_city)

                new_state = State(
                    current_city=neighbor,
                    visited=new_visited,
                    path=new_path,
                    g=new_g,
                    h=new_h
                )
                heapq.heappush(open_list, new_state)

    return None, float("inf")


if __name__ == "__main__":
    # Đồ thị khoảng cách giữa các điểm giao hàng
    graph = {
        'Kho':   {'Kho': 0, 'A': 10, 'B': 15, 'C': 20, 'D': 25},
        'A':     {'Kho': 10, 'A': 0, 'B': 35, 'C': 25, 'D': 17},
        'B':     {'Kho': 15, 'A': 35, 'B': 0, 'C': 30, 'D': 28},
        'C':     {'Kho': 20, 'A': 25, 'B': 30, 'C': 0, 'D': 16},
        'D':     {'Kho': 25, 'A': 17, 'B': 28, 'C': 16, 'D': 0}
    }

    start_city = 'Kho'

    path, cost = a_star_delivery(graph, start_city)

    if path:
        print("Lộ trình giao hàng tối ưu:")
        print(" -> ".join(path))
        print("Tổng chi phí:", cost)
    else:
        print("Không tìm được lộ trình.")

Lộ trình giao hàng tối ưu:
Kho -> A -> D -> C -> B -> Kho
Tổng chi phí: 88
